# BREADS MIRI Tutorial 3:  Building forward modeled data cubes (ish)


<div class="alert alert-block alert-info">
This is the third in a series of notebooks demonstrating data reduction for MIRI using BREADS. 
    <OL><LI>Tutorial 1: Pipeline data reductions to get ready for forward modeling</LI>
        <LI>Tutorial 2: Forward modeling and measuring SNR of a companion</LI>
        <LI>Tutorial 3 (This notebook): Generating data-cube-like representations of the forward modeled data. </LI>
</div>


<div class="alert alert-block alert-warning">
Work in progress. This notebook needs more explanations and pedagogy. 
</div>


In [1]:
import os

import numpy as np
from astropy.io import fits


# Print out what pipeline version we're using
from breads.jwst_tools.reduction_utils import find_files_to_process
from breads.jwst_tools.reduction_utils import compute_normalized_stellar_spectrum_miri
from breads.jwst_tools.reduction_utils import compute_starlight_subtraction_miri
from breads.jwst_tools.reduction_utils import get_combined_regwvs_miri
from breads.instruments.jwstmiri_cal import build_cube, build_cube_para

import multiprocessing as mp

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]= "1"


os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'


In [2]:

def create_miri_nodes_wave_sampling(channel_band, N_nodes):
    """ Creates wavelength sampling and sets aperture for photometry estimation
    and FOV for STPSF computation 

    Using smaller FOVs and apertures here is an optimization to keep the runtime from being too slow
    """
    if channel_band == '1A':
        x_nodes = np.linspace(4.89, 5.75, N_nodes, endpoint=True)
        wave = np.linspace(4.89, 5.75, 1050, endpoint=True)
        aperture = 0.2
        fov = 0.6
    elif channel_band == '1B':
        x_nodes = np.linspace(5.65, 6.64, N_nodes, endpoint=True)
        wave = np.linspace(5.65, 6.64, 1050, endpoint=True)
        aperture = 0.2
        fov = 0.6
    elif channel_band == '1C':
        x_nodes = np.linspace(6.51, 7.67, N_nodes, endpoint=True)
        wave = np.linspace(6.51, 7.67, 1050, endpoint=True)
        aperture = 0.2
        fov = 0.6        
    elif channel_band == '2A':
        x_nodes = np.linspace(7.47, 8.80, N_nodes, endpoint=True)
        wave = np.linspace(7.47, 8.80, 1050, endpoint=True)
        aperture = 0.2
        fov = 0.8
    elif channel_band == '2B':
        x_nodes = np.linspace(8.66, 10.14, N_nodes, endpoint=True)
        wave = np.linspace(8.66, 10.14, 1050, endpoint=True)
        aperture = 0.3
        fov = 0.8

    elif channel_band == '2C':
        x_nodes = np.linspace(10.00, 11.712, N_nodes, endpoint=True)
        wave = np.linspace(10.00, 11.712, 1050, endpoint=True)
        aperture = 0.3
        fov = 0.8
    elif channel_band == '3A':
        x_nodes = np.linspace(11.55, 13.47, N_nodes, endpoint=True)
        wave = np.linspace(11.55, 13.47, 1050, endpoint=True)
        aperture = 0.4
        fov = 0.9
    elif channel_band == '3B':
        x_nodes = np.linspace(13.34, 15.56, N_nodes, endpoint=True)
        wave = np.linspace(13.34, 15.56, 1050, endpoint=True)
        aperture = 0.4
        fov = 0.9
    elif channel_band == '3C':
        x_nodes = np.linspace(15.3, 18.1, N_nodes, endpoint=True)
        wave = np.linspace(15.3, 18.1, 1050, endpoint=True)
        aperture = 0.4
        fov = 0.9
    elif channel_band == '4A':
        x_nodes = np.linspace(17.7, 20.94, N_nodes, endpoint=True)
        wave = np.linspace(17.7, 20.94, 1050, endpoint=True)
        aperture = 0.5
        fov = 0.9
    elif channel_band == '4B':
        x_nodes = np.linspace(20.69, 24.48, N_nodes, endpoint=True)
        wave = np.linspace(20.69, 24.48, 1050, endpoint=True)
        aperture = 0.5
        fov = 0.9
    else:
        raise NotImplementedError

    return x_nodes, wave, aperture, fov


In [3]:
!ls

BT-settl_MIR.hdf5
CRDS_file
data
fig_fringes_jw01294003001_03102_00001_mirifushort_rate.png
fig_fringes_jw01294003001_03102_00002_mirifushort_rate.png
fig_fringes_jw01294003001_03102_00003_mirifushort_rate.png
fig_fringes_jw01294003001_03102_00004_mirifushort_rate.png
Tutorial_MIRI-Cube ish build.ipynb
Tutorial_MIRI-MRS_Forward_Modeling_and_SNR.ipynb
Tutorial_MIRI-MRS_Reductions_for_BREADS.ipynb
Untitled.ipynb


In [4]:

numthreads = 30

# Main directory for the data and reduced products
uncaldir = "data"
crds_dir = "./CRDS_file"
os.environ['CUSTOM_CRDS_PATH'] = crds_dir
targetname = '* bet Pic'
targetname_simbad = 'Beta Pic'
channel = '1'
band = '1A'
obs_band = '12A'

N_nodes = 40

output_dir = os.path.join(uncaldir, targetname, "cube_outputs", band)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

ra_vec = np.arange(-2.5, 2.5, 0.13)
dec_vec = np.arange(-2.5, 2.5, 0.13)


In [5]:

filename_filter = f"*_cal.fits"
stage2 = os.path.join(uncaldir, targetname, obs_band, "stage2")
print(f"Searching cal files in {stage2}")
utils_dir = os.path.join(uncaldir, targetname, f"utils_fm_{N_nodes}_nodes", band)
cleaned_cal_files = find_files_to_process(stage2, filetype=filename_filter)


Searching cal files in data/* bet Pic/12A/stage2
Searching in data/* bet Pic/12A/stage2 for files matching jw*_*_cal.fits
	Found 4 input files to process
	jw01294003001_03102_00001_mirifushort_cal.fits
	jw01294003001_03102_00002_mirifushort_cal.fits
	jw01294003001_03102_00003_mirifushort_cal.fits
	jw01294003001_03102_00004_mirifushort_cal.fits


In [6]:

hdulist_sc = fits.open(cleaned_cal_files[0])
wave2d = hdulist_sc["WAVELENGTH"].data

wv_nodes, wv_sampling, aperture, fov = create_miri_nodes_wave_sampling(band, N_nodes)

splitbasename = os.path.basename(cleaned_cal_files[0]).split("_")
filename_suffix = "_webbpsf"
cube_filename = os.path.join(output_dir, splitbasename[0]+"_"+splitbasename[1]+"_"+splitbasename[3]+"_spectral_cube_ish.fits")


In [7]:

mypool = mp.Pool(processes=numthreads)

##############
## Compute normalized spectrum of the star in the FOV
print('Extract star spectrum')
print(wv_nodes)
combined_star_func = compute_normalized_stellar_spectrum_miri(cleaned_cal_files, channel, utils_dir, wave2d, target_name=targetname_simbad, fit_centroid=True,
                                        wv_nodes=wv_nodes,
                                        mask_charge_transfer_radius = None,
                                        mppool=mypool,
                                        ra_dec_point_sources=None, overwrite=True)


Extract star spectrum
[4.89       4.91205128 4.93410256 4.95615385 4.97820513 5.00025641
 5.02230769 5.04435897 5.06641026 5.08846154 5.11051282 5.1325641
 5.15461538 5.17666667 5.19871795 5.22076923 5.24282051 5.26487179
 5.28692308 5.30897436 5.33102564 5.35307692 5.37512821 5.39717949
 5.41923077 5.44128205 5.46333333 5.48538462 5.5074359  5.52948718
 5.55153846 5.57358974 5.59564103 5.61769231 5.63974359 5.66179487
 5.68384615 5.70589744 5.72794872 5.75      ]
data/* bet Pic/12A/stage2/jw01294003001_03102_00001_mirifushort_cal.fits
Reading data from data/* bet Pic/12A/stage2/jw01294003001_03102_00001_mirifushort_cal.fits
Wavelength map loaded
	Unpacking data quality bitmasks. DQ array is of type uint32
Loading data for compute_med_filt_badpix cached in data/* bet Pic/utils_fm_40_nodes/1A/jw01294003001_03102_00001_mirifushort_cal_roughbadpix.fits
Loading data for compute_coordinates_arrays cached in data/* bet Pic/utils_fm_40_nodes/1A/jw01294003001_03102_00001_mirifushort_cal_relcoo

In [8]:

##############
## Fit the normalized spectrum of the star everywhere by modulating the continuum
# This does a few things:
# - subtracts the starlight everywhere as kind of fancy high-pass filter. New files saved in [utils_dir]/starsub1d.
# - Updates the bad pixel map in the original data object with a sigma clipping
print('Subtract star spectrum')
dataobj_list = compute_starlight_subtraction_miri(cleaned_cal_files, channel, utils_dir, crds_dir, wave2d, combined_star_func=combined_star_func,
                                  coords_offset=None, wv_nodes=wv_nodes, mppool=mypool)


Subtract star spectrum
data/* bet Pic/12A/stage2/jw01294003001_03102_00001_mirifushort_cal.fits
Reading data from data/* bet Pic/12A/stage2/jw01294003001_03102_00001_mirifushort_cal.fits
Wavelength map loaded
	Unpacking data quality bitmasks. DQ array is of type uint32
Loading data for compute_med_filt_badpix cached in data/* bet Pic/utils_fm_40_nodes/1A/jw01294003001_03102_00001_mirifushort_cal_roughbadpix.fits
Loading data for compute_coordinates_arrays cached in data/* bet Pic/utils_fm_40_nodes/1A/jw01294003001_03102_00001_mirifushort_cal_relcoords.fits
Running convert_MJy_per_sr_to_MJy with parameters:
	 save_utils: True
Running apply_coords_offset with parameters:
	 save_utils: True
	 coords_offset: None
Applying relative coordinate offset [0, 0]
data/* bet Pic/12A/stage2/jw01294003001_03102_00002_mirifushort_cal.fits
Reading data from data/* bet Pic/12A/stage2/jw01294003001_03102_00002_mirifushort_cal.fits
Wavelength map loaded
	Unpacking data quality bitmasks. DQ array is of typ

In [9]:

##############
## Interpolation the wavelength dimension onto a regular wavelength grid and return combined dataset object
regwvs_combdataobj = get_combined_regwvs_miri(dataobj_list, channel, wv_sampling=wv_sampling,
                                             use_starsub1d=True)


starsub1d path for combined regwvs miri: data/* bet Pic/utils_fm_40_nodes/1A/starsub1d/jw01294003001_03102_00001_mirifushort_cal.fits
Reading data from data/* bet Pic/utils_fm_40_nodes/1A/starsub1d/jw01294003001_03102_00001_mirifushort_cal.fits
Wavelength map loaded
	Unpacking data quality bitmasks. DQ array is of type uint32
regwvs path for combined regwvs miri: data/* bet Pic/utils_fm_40_nodes/1A/jw01294003001_03102_00001_mirifushort_cal_starsub1d_regwvs.fits
[DEBUG] get combined regwvs miri checking wv_sampling (1050,)
starsub1d path for combined regwvs miri: data/* bet Pic/utils_fm_40_nodes/1A/starsub1d/jw01294003001_03102_00002_mirifushort_cal.fits
Reading data from data/* bet Pic/utils_fm_40_nodes/1A/starsub1d/jw01294003001_03102_00002_mirifushort_cal.fits
Wavelength map loaded
	Unpacking data quality bitmasks. DQ array is of type uint32
regwvs path for combined regwvs miri: data/* bet Pic/utils_fm_40_nodes/1A/jw01294003001_03102_00002_mirifushort_cal_starsub1d_regwvs.fits
[DEBUG

In [10]:

# Fit a model PSF (WebbPSF) to the combined point cloud of dataobj_list
# Save output as fitpsf_filename
init_centroid = (0, 0)
ann_width = None
padding = 0.0
sector_area = None
debug_init,debug_end = None, None


In [11]:


# Load the webbPSF model (or compute if it does not yet exist)
webbpsf_reload = regwvs_combdataobj.reload_webbpsf_model()
if webbpsf_reload is None:
    webbpsf_reload = regwvs_combdataobj.compute_webbpsf_model(
            wv_sampling=wv_sampling,
            image_mask=None,
            oversample=10,
            parallelize=False, fov=fov,
            save_utils=True)


Computing PSFs. This has to iterate over many wavelengths, so is slow.
Loading telescope state as of observation date
iterating query, tdelta=3.0

MAST OPD query around UTC: 2023-01-11T05:11:27.552
                        MJD: 59955.21629111111

OPD immediately preceding the given datetime:
	URI:	 mast:JWST/product/R2023011003-NRCA3_FP1-1.fits
	Date (MJD):	 59953.6985
	Delta time:	 -1.5178 days

OPD immediately following the given datetime:
	URI:	 mast:JWST/product/R2023011203-NRCA3_FP1-1.fits
	Date (MJD):	 59955.7773
	Delta time:	 0.5610 days
User requested choosing OPD time closest in time to 2023-01-11T05:11:27.552, which is R2023011203-NRCA3_FP1-1.fits, delta time 0.561 days
Importing and format-converting OPD from /Users/mperrin/software/webbpsf-data/MAST_JWST_WSS_OPDs/R2023011203-NRCA3_FP1-1.fits
Backing out SI WFE and OTE field dependence at the WF sensing field point (NRCA3_FP1)
Current index of wavelength 0, Current wavelength 4.89, Total number of wavelength 1050
Current inde

In [12]:

wpsfs, wpsfs_header, wepsfs, webbpsf_wvs, webbpsf_X, webbpsf_Y, wpsf_oversample, wpsf_pixelscale = webbpsf_reload
webbpsf_X = np.tile(webbpsf_X[None, :, :], (wepsfs.shape[0], 1, 1))
webbpsf_Y = np.tile(webbpsf_Y[None, :, :], (wepsfs.shape[0], 1, 1))

flux_cube, fluxerr_cube, ra_grid, dec_grid = build_cube_para(regwvs_combdataobj, # combined point cloud
                   wepsfs, webbpsf_X, webbpsf_Y, # webbPSF model for flux extraction
                   ra_vec, dec_vec, # spatial sampling of final cube
                   out_filename=cube_filename,linear_interp=True,mppool=mypool,aper_radius=aperture, N_pix_min=1,
                   debug_init=debug_init, debug_end=debug_end)  # min max wavelength indices for partial extraction



[LOG] BUILD CUBE MIRI PARA
Setting parallel_flag = True
(4128, 1050) -3.6963767299336947 2.5502054718854867
Processing wavelength indices in range: 0 to 1050
prepping build_cube inputs... id: 1049 wave: 5.75starting parallel _build_cube_task...


100%|███████████████████████████████████████████████████████████| 1050/1050 [00:31<00:00, 33.65it/s]

cubing outputs... id: 271 wave: 5.1121734985700665

cubing outputs... id: 1049 wave: 5.75saving data/* bet Pic/cube_outputs/1A/jw01294003001_03102_mirifushort_spectral_cube_ish.fits
